# Axiom Router Distillation (TensorFlow)

Trains a tiny text classifier that routes science questions by scoped experiment type and subject:
`query -> { artifact_type, domain, complexity }`, distilled from the production LLM Router.

**How to run:** Runtime > Run all. When the upload widget appears, pick `train.jsonl` and `test.jsonl`
from `ml/router-distill/dataset/clean/` (produced by `generate_dataset.py` + `prepare_dataset.py`).
CPU runtime is plenty; training takes under a minute.

Success criteria (ml/router-distill/README.md): artifact_type acc >= 0.90, domain acc >= 0.92,
text_only recall >= 0.95, model < 5 MB.

In [ ]:
import json, os
import numpy as np
import tensorflow as tf
print('TF', tf.__version__)

if not (os.path.exists('train.jsonl') and os.path.exists('test.jsonl')):
    from google.colab import files
    uploaded = files.upload()  # pick train.jsonl AND test.jsonl

def load(path):
    return [json.loads(l) for l in open(path, encoding='utf-8') if l.strip()]

train, test = load('train.jsonl'), load('test.jsonl')
print(len(train), 'train rows /', len(test), 'test rows')
train[:2]

In [ ]:
ARTIFACT_TYPES = ['simulation', 'explorable_diagram', 'virtual_experiment', 'data_visualization', 'text_only']
DOMAINS = ['physics', 'chemistry', 'biology', 'earth_space', 'math_adjacent']
COMPLEXITIES = [1, 2, 3]

def xy(subset):
    x = np.array([r['query'] for r in subset], dtype=object)
    y = {
        'artifact_type': np.array([ARTIFACT_TYPES.index(r['artifact_type']) for r in subset]),
        'domain':        np.array([DOMAINS.index(r['domain']) for r in subset]),
        'complexity':    np.array([COMPLEXITIES.index(r['complexity']) for r in subset]),
    }
    return x, y

x_train, y_train = xy(train)
x_test, y_test = xy(test)

In [ ]:
vectorize = tf.keras.layers.TextVectorization(max_tokens=8000, output_sequence_length=32, name='vectorize')
vectorize.adapt(x_train)

inp = tf.keras.Input(shape=(1,), dtype=tf.string, name='query')
x = vectorize(inp)
x = tf.keras.layers.Embedding(8000, 64, name='embed')(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = {
    'artifact_type': tf.keras.layers.Dense(len(ARTIFACT_TYPES), activation='softmax', name='artifact_type')(x),
    'domain':        tf.keras.layers.Dense(len(DOMAINS), activation='softmax', name='domain')(x),
    'complexity':    tf.keras.layers.Dense(len(COMPLEXITIES), activation='softmax', name='complexity')(x),
}
model = tf.keras.Model(inp, outputs)
model.compile(optimizer='adam',
              loss={k: 'sparse_categorical_crossentropy' for k in outputs},
              metrics={k: ['accuracy'] for k in outputs})
model.summary()

In [ ]:
history = model.fit(
    x_train, y_train,
    validation_split=0.1, epochs=40, batch_size=32, verbose=2,
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
)

In [ ]:
results = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
preds = model.predict(x_test, verbose=0)
type_pred = preds['artifact_type'].argmax(axis=1)
ti = ARTIFACT_TYPES.index('text_only')
mask = y_test['artifact_type'] == ti
metrics = {
    'held_out_n': len(test),
    'artifact_type_acc': round(results['artifact_type_accuracy'], 4),
    'domain_acc': round(results['domain_accuracy'], 4),
    'complexity_acc': round(results['complexity_accuracy'], 4),
    'text_only_recall': round(float((type_pred[mask] == ti).mean()), 4) if mask.any() else None,
}
print(json.dumps(metrics, indent=2))

# spot check against known-answer queries from the golden set
for q in ['why does ice float?', 'show me projectile motion', 'what is the boiling point of gold?', 'write me a poem about clouds']:
    p = model.predict(np.array([q], dtype=object), verbose=0)
    print(f"{q!r:55} -> {ARTIFACT_TYPES[p['artifact_type'].argmax()]:20} {DOMAINS[p['domain'].argmax()]:14} conf={p['artifact_type'].max():.2f}")

In [ ]:
os.makedirs('out', exist_ok=True)
model.save('out/router_student.keras')
model.export('out/saved_model')
json.dump(metrics, open('out/metrics.json', 'w'), indent=2)
json.dump({'artifact_type': ARTIFACT_TYPES, 'domain': DOMAINS, 'complexity': COMPLEXITIES}, open('out/labels.json', 'w'))
try:
    converter = tf.lite.TFLiteConverter.from_saved_model('out/saved_model')
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
    open('out/router_student.tflite', 'wb').write(converter.convert())
    print('tflite exported')
except Exception as e:
    print('tflite export skipped:', e)

!zip -qr router_student_out.zip out
try:
    from google.colab import files
    files.download('router_student_out.zip')
except ImportError:
    print('not on Colab; artifacts are in ./out')